# Learning Tensorflow

In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.regularizers import l2


from sklearn.preprocessing import MinMaxScaler
import plotly.express as px
from plotly.subplots import make_subplots
import random
import numpy as np


In [2]:
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  raise SystemError('GPU device not found')

print('Found GPU at: {}'.format(device_name))

SystemError: GPU device not found

In [3]:
device_name

''

In [4]:
df = mnist.load_data()


In [5]:
X_train, y_train = df[0]
X_test, y_test = df[1]

X_train = X_train / 255.0
X_test = X_test / 255.0

In [6]:
X_train_flattened = X_train.reshape(X_train.shape[0], -1)
X_test_flattened = X_test.reshape(X_test.shape[0], -1)

In [22]:
mnist_model = Sequential(
    [
        Input(shape=(784,)),

        Dropout(0.2),

        Dense(500, activation='relu'),

        Dropout(0.2),

        Dense(200, activation='relu'),
        Dropout(0.2),
        Dense(100, activation='relu'),
        Dropout(0.2),
        Dense(50, activation='relu'),
        
        Dense(10, activation='linear' )
    ],
    name = "MNIST_Neural_Network"
)


mnist_model.compile(
    optimizer=Adam(learning_rate=0.0004
                   ),
     loss=SparseCategoricalCrossentropy(from_logits=True),
     metrics=['accuracy'])
mnist_model.summary()

Model: "MNIST_Neural_Network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dropout_4 (Dropout)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 500)            │       392,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 500)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 200)            │       100,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 100)            │        20,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 10)             │           510 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 518,360 (1.98 MB)

 Trainable params: 518,360 (1.98 MB)

 Non-trainable params: 0 (0.00 B)

# Training

In [24]:
with tf.device(device_name):
  history = mnist_model.fit(X_train_flattened, y_train,
                          epochs=20,
                          verbose=1,
                          batch_size=700,
                          validation_data =(X_test_flattened, y_test))

Epoch 1/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9854 - loss: 0.0444 - val_accuracy: 0.9854 - val_loss: 0.0506
Epoch 2/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9868 - loss: 0.0415 - val_accuracy: 0.9851 - val_loss: 0.0519
Epoch 3/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9873 - loss: 0.0399 - val_accuracy: 0.9849 - val_loss: 0.0519
Epoch 4/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9873 - loss: 0.0386 - val_accuracy: 0.9846 - val_loss: 0.0540
Epoch 5/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9884 - loss: 0.0383 - val_accuracy: 0.9861 - val_loss: 0.0523
Epoch 6/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9875 - loss: 0.0373 - val_accuracy: 0.9860 - val_loss: 0.0537
Epoch 7/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9886 - loss: 0.0358 - val_accuracy: 0.9844 - val_loss: 0.0551
Epoch 8/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9881 - loss: 0.0375 - val_accuracy: 0.9864 - v

In [25]:
def plot_results(start_epoch=0):
    acc= history.history['accuracy'][start_epoch:]
    loss = history.history['loss'][start_epoch:]
    val_acc = history.history['val_accuracy'][start_epoch:]
    val_loss = history.history['val_loss'][start_epoch:]

    fig = make_subplots(2,2, subplot_titles=("Accuracy", "Loss", "Val Accuracy", "Val Loss"))

    fig.add_trace(px.line(x=history.epoch[start_epoch:], y=acc).data[0], 1, 1)
    fig.add_trace(px.line(x=history.epoch[start_epoch:], y=loss).data[0], 1, 2)
    fig.add_trace(px.line(x=history.epoch[start_epoch:], y=val_acc).data[0], 2, 1)
    fig.add_trace(px.line(x=history.epoch[start_epoch:], y=val_loss).data[0], 2, 2)

    fig.show()

def print_results():
  predictions = mnist_model.predict(X_test_flattened)
  predicted_labels = tf.argmax(predictions, axis=1).numpy()
  print( "######################################## Results ################################" )
  for i in history.history.keys():
    print(f'{i}:                           {history.history[i][-1]: .5f}  ')

  misclassified_indices = np.where(predicted_labels != y_test)[0]
  num_misclassified = len(misclassified_indices)
  print(f'Missclassified: {num_misclassified}')

print_results()
plot_results(0)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
######################################## Results ################################
accuracy:                            0.99105  
loss:                            0.02686  
val_accuracy:                            0.98620  
val_loss:                            0.05222  
Missclassified: 138


In [26]:
def test( num:int=0 ):
  test_sample = X_test[num]
  actual_label = y_test[num]
  scaler = MinMaxScaler()

  pred_array = mnist_model.predict(test_sample.reshape(1, -1))[0]
  pred_array_scaled = scaler.fit_transform(pred_array.reshape(-1, 1)).reshape(-1,)
  pred_num = tf.argmax(pred_array).numpy()
  confidence = pred_array_scaled[pred_num]

  fig = make_subplots(1,2)
  label_image = px.imshow( test_sample[::-1,] , title=f'Actual Label: {actual_label}')
  pred_histo = px.histogram( x=range(10), y=pred_array_scaled, nbins=10 )

  fig.add_trace( label_image.data[0], row=1, col=1 )
  fig.add_trace( pred_histo.data[0], row=1, col=2 )
  fig.update_layout( title_text=f'Actual Label: {actual_label}                       Prediction: {pred_num}\
  confidence: {confidence*100:8f} %')

  #show sample
  return fig.show()

In [27]:

def test_random(times:int = 1):
  for i in range(times):
    test( random.randint(0, 10_000) )

In [28]:
test_random(5)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step


In [29]:
# Save the model in the SavedModel format (recommended)
mnist_model.save('Dense_Model.keras')
